<a href="https://colab.research.google.com/github/rodrigoesborges/conju-python/blob/main/GRUPO_4_(Pol%C3%ADtica_Monet%C3%A1ria)_VOLUME_DE_CR%C3%89DITO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [91]:
!pip install python-bcb -q

In [92]:
import pandas as pd
from bcb import sgs

In [93]:
series_credito = {
    'Crédito Total': 20539,
    'Pessoa Jurídica (PJ) - Total': 20540,
    'PJ - Recursos Livres': 20543,
    'PJ - Recursos Direcionados': 20594,
    'Pessoa Física (PF) - Total': 20541,
    'PF - Recursos Livres': 20570,
    'PF - Recursos Direcionados': 20606,
    'PF - Crédito Rural': 20607,
    'PF - Financiamento Imobiliário': 20610
}

print(f"Total de séries mapeadas: {len(series_credito)}")

Total de séries mapeadas: 9


In [94]:
list(series_credito.keys())

['Crédito Total',
 'Pessoa Jurídica (PJ) - Total',
 'PJ - Recursos Livres',
 'PJ - Recursos Direcionados',
 'Pessoa Física (PF) - Total',
 'PF - Recursos Livres',
 'PF - Recursos Direcionados',
 'PF - Crédito Rural',
 'PF - Financiamento Imobiliário']

In [95]:
df_bruto = sgs.get(series_credito, start='2024-01-01')
df_bruto.head()

,Crédito Total,Pessoa Jurídica (PJ) - Total,PJ - Recursos Livres,PJ - Recursos Direcionados,Pessoa Física (PF) - Total,PF - Recursos Livres,PF - Recursos Direcionados,PF - Crédito Rural,PF - Financiamento Imobiliário
Date,,,,,,,,,
2024-01-01,5794707,2234362,1421476,812886,3560345,1947518,1612827,209272,102284
2024-02-01,5815954,2235302,1423312,811990,3580652,1953857,1626796,215763,103079
2024-03-01,5899742,2285775,1470733,815043,3613967,1969433,1644534,223049,104355
2024-04-01,5915538,2268879,1452275,816604,3646659,1989763,1656896,227186,105461
2024-05-01,5953960,2278309,1453046,825262,3675652,2008616,1667035,230154,107201


In [96]:
print("Linhas e colunas:", df_bruto.shape)
df_bruto.tail(3)

Linhas e colunas: (33, 9)


,Crédito Total,Pessoa Jurídica (PJ) - Total,PJ - Recursos Livres,PJ - Recursos Direcionados,Pessoa Física (PF) - Total,PF - Recursos Livres,PF - Recursos Direcionados,PF - Crédito Rural,PF - Financiamento Imobiliário
Date,,,,,,,,,
2026-07-01,7362252,2729062,1585565,1143497,4633191,2551651,2081540,240541,167762
2026-08-01,7412361,2731883,1577667,1154216,4680478,2577010,2103468,237696,169272
2026-09-01,7444208,2759442,1605274,1154169,4684766,2582163,2102603,226565,170988


In [97]:
df_bruto.isna().sum()

,0
Crédito Total,0
Pessoa Jurídica (PJ) - Total,0
PJ - Recursos Livres,0
PJ - Recursos Direcionados,0
Pessoa Física (PF) - Total,0
PF - Recursos Livres,0
PF - Recursos Direcionados,0
PF - Crédito Rural,0
PF - Financiamento Imobiliário,0


In [98]:
def criar_tabela_base(df):
    base = df.reset_index().melt(
        id_vars='Date',
        var_name='indicador',
        value_name='valor'
    )
    base['data'] = base['Date'].dt.strftime('%Y%m01')
    base = base.sort_values(['indicador', 'Date']).reset_index(drop=True)
    return base

In [99]:
tabela_base = criar_tabela_base(df_bruto)
tabela_base.head(10)

,Date,indicador,valor,data
0,2024-01-01,Crédito Total,5794707,20240101
1,2024-02-01,Crédito Total,5815954,20240201
2,2024-03-01,Crédito Total,5899742,20240301
3,2024-04-01,Crédito Total,5915538,20240401
4,2024-05-01,Crédito Total,5953960,20240501
5,2024-06-01,Crédito Total,6041898,20240601
6,2024-07-01,Crédito Total,6069205,20240701
7,2024-08-01,Crédito Total,6135298,20240801
8,2024-09-01,Crédito Total,6216528,20240901
9,2024-10-01,Crédito Total,6275117,20241001


In [100]:
tabela_base[['data', 'indicador', 'valor']].tail(9)

,data,indicador,valor
288,20260101,Pessoa Jurídica (PJ) - Total,2654220
289,20260201,Pessoa Jurídica (PJ) - Total,2656484
290,20260301,Pessoa Jurídica (PJ) - Total,2687535
291,20260401,Pessoa Jurídica (PJ) - Total,2685397
292,20260501,Pessoa Jurídica (PJ) - Total,2708154
293,20260601,Pessoa Jurídica (PJ) - Total,2750759
294,20260701,Pessoa Jurídica (PJ) - Total,2729062
295,20260801,Pessoa Jurídica (PJ) - Total,2731883
296,20260901,Pessoa Jurídica (PJ) - Total,2759442


In [111]:
def calcular_var_ano(base):
    df = base.copy()
    df['ano'] = df['Date'].dt.year

    dezembro = df[df['Date'].dt.month == 12][['indicador', 'ano', 'valor']].copy()
    dezembro['ano'] = dezembro['ano'] + 1
    dezembro = dezembro.rename(columns={'valor': 'valor_dez_ant'})

    df = df.merge(dezembro, on=['indicador', 'ano'], how='left')
    df['valor'] = ((df['valor'] / df['valor_dez_ant']) - 1) * 100

    return df[['data', 'indicador', 'valor']].dropna()

In [102]:
tab_interm_1 = calcular_var_ano(tabela_base)
tab_interm_1.tail(9)

,data,indicador,valor
288,20260101,Pessoa Jurídica (PJ) - Total,-1.758685
289,20260201,Pessoa Jurídica (PJ) - Total,-1.674887
290,20260301,Pessoa Jurídica (PJ) - Total,-0.525588
291,20260401,Pessoa Jurídica (PJ) - Total,-0.604723
292,20260501,Pessoa Jurídica (PJ) - Total,0.237588
293,20260601,Pessoa Jurídica (PJ) - Total,1.814538
294,20260701,Pessoa Jurídica (PJ) - Total,1.011461
295,20260801,Pessoa Jurídica (PJ) - Total,1.115876
296,20260901,Pessoa Jurídica (PJ) - Total,2.135924


In [112]:
def calcular_var_12m(base):
    df = base.copy()
    df['valor'] = df.groupby('indicador')['valor'].pct_change(periods=12) * 100
    return df[['data', 'indicador', 'valor']].dropna()

In [104]:
tab_interm_2 = calcular_var_12m(tabela_base)
tab_interm_2.tail(9)

,data,indicador,valor
288,20260101,Pessoa Jurídica (PJ) - Total,8.249044
289,20260201,Pessoa Jurídica (PJ) - Total,7.239486
290,20260301,Pessoa Jurídica (PJ) - Total,7.576187
291,20260401,Pessoa Jurídica (PJ) - Total,6.884621
292,20260501,Pessoa Jurídica (PJ) - Total,6.927210
293,20260601,Pessoa Jurídica (PJ) - Total,7.970584
294,20260701,Pessoa Jurídica (PJ) - Total,7.257585
295,20260801,Pessoa Jurídica (PJ) - Total,7.289815
296,20260901,Pessoa Jurídica (PJ) - Total,6.549691


In [105]:
ultimas_3_datas = sorted(tabela_base['data'].unique())[-3:]
print("Últimas 3 datas:", ultimas_3_datas)

Últimas 3 datas: ['20260701', '20260801', '20260901']


In [106]:
def montar_painel_wide(base, t_ano, t_12m, ordem_categorias):
    ultimas = sorted(base['data'].unique())[-3:]
    ultima_data = ultimas[-1]

    wide = base[base['data'].isin(ultimas)].pivot(
        index='indicador',
        columns='data',
        values='valor'
    )

    col_ano = t_ano[t_ano['data'] == ultima_data].set_index('indicador')[['valor']]
    col_ano = col_ano.rename(columns={'valor': f'Var. no ano ({ultima_data}) %'})

    col_12m = t_12m[t_12m['data'] == ultima_data].set_index('indicador')[['valor']]
    col_12m = col_12m.rename(columns={'valor': f'Var. 12m ({ultima_data}) %'})

    painel = wide.join([col_ano, col_12m])
    return painel.reindex(ordem_categorias).round(2)

In [107]:
ordem = list(series_credito.keys())
painel_final = montar_painel_wide(tabela_base, tab_interm_1, tab_interm_2, ordem)

In [108]:
display(painel_final)

,20260701,20260801,20260901,Var. no ano (20260901) %,Var. 12m (20260901) %
indicador,,,,,
Crédito Total,7362252,7412361,7444208,4.30,8.63
Pessoa Jurídica (PJ) - Total,2729062,2731883,2759442,2.14,6.55
PJ - Recursos Livres,1585565,1577667,1605274,-2.04,1.22
PJ - Recursos Direcionados,1143497,1154216,1154169,8.58,14.97
Pessoa Física (PF) - Total,4633191,4680478,4684766,5.62,9.90
PF - Recursos Livres,2551651,2577010,2582163,5.02,10.03
PF - Recursos Direcionados,2081540,2103468,2102603,6.37,9.73
PF - Crédito Rural,240541,237696,226565,-1.49,-1.21
PF - Financiamento Imobiliário,167762,169272,170988,9.70,16.98


In [109]:
data_ref = ultimas_3_datas[-1]
total_cred = painel_final.loc['Crédito Total', data_ref]
total_pf = painel_final.loc['Pessoa Física (PF) - Total', data_ref]
total_pj = painel_final.loc['Pessoa Jurídica (PJ) - Total', data_ref]

print(f"Data de referência: {data_ref}")
print(f"Participação PF: {(total_pf / total_cred) * 100:.2f}%")
print(f"Participação PJ: {(total_pj / total_cred) * 100:.2f}%")

Data de referência: 20260901
Participação PF: 62.93%
Participação PJ: 37.07%


In [110]:
!pip install Plotly

In [113]:
import plotly.express as px
import plotly.graph_objects as go

In [117]:
principais = ['Crédito Total', 'Pessoa Física (PF) - Total', 'Pessoa Jurídica (PJ) - Total']
df_linhas = tabela_base[tabela_base['indicador'].isin(principais)].copy()
df_linhas.tail(6)

,Date,indicador,valor,data
291,2026-04-01,Pessoa Jurídica (PJ) - Total,2685397,20260401
292,2026-05-01,Pessoa Jurídica (PJ) - Total,2708154,20260501
293,2026-06-01,Pessoa Jurídica (PJ) - Total,2750759,20260601
294,2026-07-01,Pessoa Jurídica (PJ) - Total,2729062,20260701
295,2026-08-01,Pessoa Jurídica (PJ) - Total,2731883,20260801
296,2026-09-01,Pessoa Jurídica (PJ) - Total,2759442,20260901


In [118]:
fig1 = px.line(
    df_linhas,
    x='Date',
    y='valor',
    color='indicador',
    markers=True,
    title='Evolução do Saldo de Crédito (R$ milhões) — Total, PF e PJ',
    labels={'Date': 'Mês', 'valor': 'Saldo (R$ milhões)', 'indicador': 'Categoria'}
)
fig1.update_layout(template='plotly_white', hovermode='x unified')
fig1.show()

In [119]:
col_var_12m = f'Var. 12m ({data_ref}) %'
df_barras = painel_final.drop('Crédito Total').sort_values(col_var_12m, ascending=True)
df_barras[[col_var_12m]]

,Var. 12m (20260901) %
indicador,
PF - Crédito Rural,-1.21
PJ - Recursos Livres,1.22
Pessoa Jurídica (PJ) - Total,6.55
PF - Recursos Direcionados,9.73
Pessoa Física (PF) - Total,9.90
PF - Recursos Livres,10.03
PJ - Recursos Direcionados,14.97
PF - Financiamento Imobiliário,16.98


In [120]:
fig2 = px.bar(
    df_barras,
    x=col_var_12m,
    y=df_barras.index,
    orientation='h',
    text=col_var_12m,
    title=f'Crescimento do Crédito em 12 Meses (%) — Ref: {data_ref}',
    labels={col_var_12m: 'Variação 12m (%)', 'indicador': 'Modalidade'}
)
fig2.update_traces(texttemplate='%{text:.2f}%', textposition='outside', marker_color='#2b6cb0')
fig2.update_layout(
    template='plotly_white',
    xaxis_range=[df_barras[col_var_12m].min() - 3, df_barras[col_var_12m].max() + 4]
)
fig2.show()

In [121]:
labels_pizza = ['Pessoa Física (PF)', 'Pessoa Jurídica (PJ)']
valores_pizza = [total_pf, total_pj]
print(labels_pizza, valores_pizza)

['Pessoa Física (PF)', 'Pessoa Jurídica (PJ)'] [np.int64(4684766), np.int64(2759442)]


In [122]:
fig3 = go.Figure(data=[go.Pie(
    labels=labels_pizza,
    values=valores_pizza,
    hole=0.45,
    textinfo='label+percent',
    marker_colors=['#2f855a', '#3182ce']
)])
fig3.update_layout(
    title=f'Participação no Saldo de Crédito Total — {data_ref}',
    template='plotly_white'
)
fig3.show()

In [123]:
col_var_ano = f'Var. no ano ({data_ref}) %'
df_comp = painel_final[[col_var_ano, col_var_12m]].reset_index()
df_comp.head()

,indicador,Var. no ano (20260901) %,Var. 12m (20260901) %
0,Crédito Total,4.30,8.63
1,Pessoa Jurídica (PJ) - Total,2.14,6.55
2,PJ - Recursos Livres,-2.04,1.22
3,PJ - Recursos Direcionados,8.58,14.97
4,Pessoa Física (PF) - Total,5.62,9.90


In [124]:
fig4 = px.bar(
    df_comp,
    x='indicador',
    y=[col_var_ano, col_var_12m],
    barmode='group',
    title=f'Comparação: Variação no Ano vs. Variação em 12 Meses ({data_ref})',
    labels={'value': 'Variação (%)', 'indicador': 'Categoria', 'variable': 'Indicador'}
)
fig4.update_layout(template='plotly_white', xaxis_tickangle=-30)
fig4.show()

In [125]:
df_traj = tab_interm_2[tab_interm_2['indicador'].isin(principais)]

fig5 = px.line(
    df_traj,
    x='data',
    y='valor',
    color='indicador',
    markers=True,
    title='Trajetória da Variação % em 12 Meses — Total, PF e PJ',
    labels={'data': 'Data', 'valor': 'Variação 12m (%)', 'indicador': 'Categoria'}
)
fig5.update_layout(template='plotly_white', xaxis_tickangle=-30)
fig5.show()